# Job A (trained models) — STFPM / RD4AD / DRAEM / CSFlow on Real-IAD

**Model-major loop:** one model at a time across all 30 categories. Per-session pick which model to run by editing `MODEL` in cell 4.

**Realistic time budget on T4 (Colab Pro):**
| Model | Per cat | × 30 cats | Sessions needed |
|---|---|---|---|
| `anomalib_stfpm` (100 ep) | ~15 min | ~7.5 h | 1 |
| `rd4ad` (200 ep)          | ~25 min | ~12.5 h | 2 |
| `anomalib_draem` (200 ep) | ~50 min | ~25 h | 3-4 |
| `anomalib_csflow` (240 ep)| ~80 min | ~40 h | **does not fit Colab — use RunPod/HPC** |

**Recommended order on Colab:** `anomalib_stfpm` → `rd4ad` → `anomalib_draem`. Skip `anomalib_csflow` here.

**Outputs:** synced to `/content/drive/MyDrive/thesis_runs/jobA_trained/` per (model × category). Resumable via `<model>__<category>.done` markers.

In [ ]:
# 1. Mount Drive and verify the Real-IAD zips are visible.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
ZIPS_DIR = '/content/drive/MyDrive/Real-IAD_dataset/realiad_1024'
zips = sorted(glob.glob(os.path.join(ZIPS_DIR, '*.zip')))
print(f'Found {len(zips)} category zips under {ZIPS_DIR}')
assert len(zips) > 0, 'No zips found — check ZIPS_DIR.'

In [ ]:
# 2. Clone (or pull) the thesis repo.
REPO_URL = 'https://github.com/<your-username>/Real-time-visual-defect-detection.git'
REPO_DIR = '/content/Real-time-visual-defect-detection'
BRANCH   = 'main'

import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--all'])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])

print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 3. Install Python dependencies. Same stack as the feature-based notebook.
# anomalib >= 2.x (module path), transformers in 4.x (avoids 5.x breaking changes),
# kornia + einops are anomalib transitive deps Colab does not always pull.
# FrEIA is required by CSFlow only — installed defensively for completeness.
!pip install -q "anomalib>=2.2,<3" lightning "transformers>=4.46,<5" scikit-learn timm kornia einops FrEIA
!pip install -q -e /content/Real-time-visual-defect-detection --no-deps || echo '[note] editable install skipped (not required)'

import torch, anomalib, lightning, transformers
print('torch       ', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('anomalib    ', anomalib.__version__)
print('lightning   ', lightning.__version__)
print('transformers', transformers.__version__)

In [ ]:
# 4. Pick the model for THIS session and launch.
#
# MODEL options (run one per session):
#   'anomalib_stfpm'   - ~7.5 h on T4, fits 1 session
#   'rd4ad'            - ~12.5 h on T4, ~2 sessions
#   'anomalib_draem'   - ~25 h on T4, ~3-4 sessions
#   'anomalib_csflow'  - ~40 h, do NOT run here (use RunPod/HPC)
#
# CATEGORIES: leave empty to run all 30. Or restrict, e.g.
#   'audiojack bottle_cap zipper'  -> only those three.

import os
os.environ['MODEL']       = 'anomalib_stfpm'
os.environ['CATEGORIES']  = ''
os.environ['ZIPS_DIR']    = '/content/drive/MyDrive/Real-IAD_dataset/realiad_1024'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/thesis_runs/jobA_trained'
os.environ['WORK_DIR']    = '/content/work'
os.environ['REPO_DIR']    = '/content/Real-time-visual-defect-detection'
os.environ['CONFIG']      = '/content/Real-time-visual-defect-detection/src/benchmark_AD/configs/colab_trained.yaml'

!mkdir -p {os.environ['RESULTS_DIR']}
!bash /content/Real-time-visual-defect-detection/scripts/run_jobA_trained_colab.sh 2>&1 | tee -a {os.environ['RESULTS_DIR']}/run_{os.environ['MODEL']}.log

## Resuming and switching models

- **Same model after disconnect:** rerun cells 1 -> 4. Categories with `<model>__<category>.done` markers are skipped automatically.
- **Switch to next model:** edit `MODEL` in cell 4 and rerun. The new model has its own `.done` markers, so it starts from category 1.
- **Restrict to a few categories** (debug or a partial sweep): set `CATEGORIES = 'audiojack bottle_cap'` in cell 4.
- **Force a single (model, category) rerun:** delete its marker on Drive, e.g.
  ```python
  !rm /content/drive/MyDrive/thesis_runs/jobA_trained/anomalib_stfpm__audiojack.done
  ```

## Troubleshooting

- **`CUDA out of memory`** on DRAEM/CSFlow: lower `anomalib.batch_size` in [colab_trained.yaml](../src/benchmark_AD/configs/colab_trained.yaml) from 8 to 4.
- **`Lightning trainer` epoch count seems wrong:** the per-model epochs live under `model.<kind>.epochs` in the config; `--model` only switches the registered name, not those values.
- **Drive I/O slow:** keeping the laptop awake and the Colab tab focused helps. The bash script streams progress so the cell stays "active" against the idle timer.

In [ ]:
# 6. Release Colab resources once the run is finished and synced.
# This clears local caches first, then disconnects the runtime.
import gc

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

gc.collect()

try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass
